In [104]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [105]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [106]:
len(words)

32033

In [107]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [108]:
# build the dataset

block_size = 3 # context length - how many charachters do we take to predict the next one
X, Y = [], []
for w in words:

    #print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [109]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [110]:
sum(p.nelement() for p in parameters)

for p in parameters:
    p.requires_grad = True

In [ ]:
iterations = 1000
lre = torch.linspace(-3, 0, iterations)
lrs = 10**lre
batch_size = 32

for iter in range(iterations):

    # batch construction
    ix = torch.randint(0, X.shape[0], (batch_size,), generator=g)

    # forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    #print(iter, loss.item())

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -learning_rate * p.grad

In [112]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
print('final loss', loss.item())

final loss 2.5214450359344482


In [113]:
P = (N+1).float()
P /= P.sum(1, keepdim=True)

NameError: name 'N' is not defined

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        ix = torch.multinomial(P[ix], num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


In [ ]:
neg_log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        
        neg_log_likelihood += -torch.log(P[ix1, ix2])
        n += 1

print ('avg_neg_log_likelihood', neg_log_likelihood.item()/n)

avg_neg_log_likelihood 2.4543562565199477


In [ ]:
# create a training set
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27).float()

In [ ]:
W = torch.randn((27, 27))
(xenc @ W).exp()

tensor([[2.8934, 4.8767, 0.1349, 2.6481, 0.1483, 3.4158, 3.1347, 8.7119, 0.9608,
         0.5438, 0.3934, 0.4900, 2.4886, 0.4237, 4.4222, 1.7326, 0.4926, 4.8204,
         2.0833, 0.6934, 0.6207, 0.5196, 0.6389, 2.0999, 1.4218, 0.4412, 0.1895],
        [1.7551, 1.9242, 0.3308, 0.4058, 2.4309, 0.5579, 0.6996, 0.3275, 2.0251,
         0.3157, 0.1917, 4.6684, 0.0792, 1.4189, 5.9486, 0.3432, 0.5766, 0.2503,
         1.5194, 2.0657, 1.3120, 0.1544, 1.5191, 0.4268, 0.5367, 5.7186, 0.8568],
        [0.6596, 1.2326, 2.1807, 0.3051, 1.3617, 0.1612, 0.3606, 0.2296, 0.2333,
         3.0942, 0.8904, 0.9911, 0.1181, 1.2339, 0.2490, 1.7740, 2.2526, 1.7356,
         1.5318, 4.6338, 0.9822, 0.6311, 0.3597, 1.9591, 2.0927, 0.6126, 0.7992],
        [0.6596, 1.2326, 2.1807, 0.3051, 1.3617, 0.1612, 0.3606, 0.2296, 0.2333,
         3.0942, 0.8904, 0.9911, 0.1181, 1.2339, 0.2490, 1.7740, 2.2526, 1.7356,
         1.5318, 4.6338, 0.9822, 0.6311, 0.3597, 1.9591, 2.0927, 0.6126, 0.7992],
        [0.3446, 1.6687,

In [ ]:
logits = xenc @ W # log-counts
counts = logits.exp() # quivalent N
probs = counts / counts.sum(1, keepdim=True)
probs

tensor([[0.0562, 0.0948, 0.0026, 0.0515, 0.0029, 0.0664, 0.0609, 0.1694, 0.0187,
         0.0106, 0.0076, 0.0095, 0.0484, 0.0082, 0.0860, 0.0337, 0.0096, 0.0937,
         0.0405, 0.0135, 0.0121, 0.0101, 0.0124, 0.0408, 0.0276, 0.0086, 0.0037],
        [0.0458, 0.0502, 0.0086, 0.0106, 0.0634, 0.0145, 0.0182, 0.0085, 0.0528,
         0.0082, 0.0050, 0.1217, 0.0021, 0.0370, 0.1551, 0.0089, 0.0150, 0.0065,
         0.0396, 0.0539, 0.0342, 0.0040, 0.0396, 0.0111, 0.0140, 0.1491, 0.0223],
        [0.0202, 0.0377, 0.0668, 0.0093, 0.0417, 0.0049, 0.0110, 0.0070, 0.0071,
         0.0947, 0.0273, 0.0303, 0.0036, 0.0378, 0.0076, 0.0543, 0.0690, 0.0531,
         0.0469, 0.1419, 0.0301, 0.0193, 0.0110, 0.0600, 0.0641, 0.0188, 0.0245],
        [0.0202, 0.0377, 0.0668, 0.0093, 0.0417, 0.0049, 0.0110, 0.0070, 0.0071,
         0.0947, 0.0273, 0.0303, 0.0036, 0.0378, 0.0076, 0.0543, 0.0690, 0.0531,
         0.0469, 0.1419, 0.0301, 0.0193, 0.0110, 0.0600, 0.0641, 0.0188, 0.0245],
        [0.0088, 0.0425,

In [ ]:
print(probs[0].sum())
print(probs[0].shape)

tensor(1.0000)
torch.Size([27])


In [ ]:
print('input', xs) # input
print('output', ys) # output

# randomly initialize W matrix
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g)

xenc = F.one_hot(xs, num_classes=27).float() # input to the network - one_hot encoded
logits = xenc @ W # predicted log-counts
# next 2 lines is a 'softmax'
counts = logits.exp() # counts - quivalent to N
probs = counts / counts.sum(1, keepdim=True) # probabilities for the next charachter
print(probs.shape)

input tensor([ 0,  5, 13, 13,  1])
output tensor([ 5, 13, 13,  1,  0])
torch.Size([5, 27])


In [ ]:
nlls = torch.zeros(5)
for i in range(5):
    # i-th bigram
    x = xs[i].item() # input charachter index
    y = ys[i].item() # output charachter index
    print('----------')
    print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
    print('input to the neural net', x)
    print('output probabilities from a neural net', probs[i])
    print('label (next charachter)', y)
    p = probs[i, y]
    print('probability assigned by the nn to the label', p.item())
    logp = torch.log(p)
    print('log likelihood', logp.item())
    nll = -logp
    print('neg log likelihood', nll.item())
    nlls[i] = nll

print('avg negative log likelehood (loss) is ', nlls.mean().item())

----------
bigram example 1: .e (indexes 0,5)
input to the neural net 0
output probabilities from a neural net tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
        0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
        0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459])
label (next charachter) 5
probability assigned by the nn to the label 0.01228625513613224
log likelihood -4.399273872375488
neg log likelihood 4.399273872375488
----------
bigram example 2: em (indexes 5,13)
input to the neural net 5
output probabilities from a neural net tensor([0.0290, 0.0796, 0.0248, 0.0521, 0.1989, 0.0289, 0.0094, 0.0335, 0.0097,
        0.0301, 0.0702, 0.0228, 0.0115, 0.0181, 0.0108, 0.0315, 0.0291, 0.0045,
        0.0916, 0.0215, 0.0486, 0.0300, 0.0501, 0.0027, 0.0118, 0.0022, 0.0472])
label (next charachter) 13
probability assigned by the nn to the label 0.018050700426101685
log likelihood -4.014570713043213
neg log 

In [ ]:
# create a training set
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples', num)

# randomly initialize W matrix
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

iterations = 100
reg_ratio = 0.01
learning_rate = 50

for run in range(iterations):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float() # input to the network - one_hot encoded
    logits = xenc @ W # predicted log-counts
    # next 2 lines is a 'softmax'
    counts = logits.exp() # counts - quivalent to N
    probs = counts / counts.sum(1, keepdim=True) # probabilities for the next charachter
    loss = -probs[torch.arange(num), ys].log().mean() + reg_ratio * (W**2).mean()
    print('loss', loss.item())

    # backward pass
    W.grad = None # set 0 to gradient
    loss.backward()
    
    # update
    W.data += -learning_rate * W.grad

number of examples 228146


loss 3.7686190605163574
loss 3.3788039684295654
loss 3.16108775138855
loss 3.02718448638916
loss 2.934483528137207
loss 2.867230176925659
loss 2.8166542053222656
loss 2.777146577835083
loss 2.745253801345825
loss 2.7188303470611572
loss 2.696505308151245
loss 2.6773719787597656
loss 2.6608052253723145
loss 2.6463513374328613
loss 2.633664846420288
loss 2.6224710941314697
loss 2.6125471591949463
loss 2.6037065982818604
loss 2.595794916152954
loss 2.5886807441711426
loss 2.5822558403015137
loss 2.5764291286468506
loss 2.5711233615875244
loss 2.566272735595703
loss 2.5618226528167725
loss 2.5577259063720703
loss 2.5539441108703613
loss 2.550442695617676
loss 2.547192335128784
loss 2.5441696643829346
loss 2.5413522720336914
loss 2.538721799850464
loss 2.536262035369873
loss 2.5339581966400146
loss 2.531797409057617
loss 2.5297679901123047
loss 2.527860164642334
loss 2.5260636806488037
loss 2.5243704319000244
loss 2.522773265838623
loss 2.521263837814331
loss 2.519836664199829
loss 2.518485

In [ ]:
# sampling from the trained neural net
g = torch.Generator().manual_seed(2147483647)

for i in range(25):

    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W # predict log-counts
        # next 2 lines is a 'softmax'
        counts = logits.exp() # counts - quivalent to N
        p = counts / counts.sum(1, keepdim=True) # probabilities for the next charachter
        
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))
        


cexze.
momasurailezityha.
konimittain.
llayn.
ka.
da.
staiyaubrtthrigotai.
moliellavo.
ke.
teda.
ka.
emim.
sade.
enkaviyny.
fobsp.
hinivenvtahlasu.
dsor.
br.
jol.
pyawaisan.
ja.
fdinee.
zka.
deru.
firit.
